# Task 7.11: audited GPU training, isolated benchmark, epoch resume
Run cells in order. Commands raise on failure. Publish the Task 7.11 fixes to Git before using this notebook; stale code is rejected. No claim of GPU readiness is made until the runtime preflight passes.

Resume restores the last **completed epoch**, including optimizer, scheduler, scaler, RNG and logs. Work in an interrupted epoch is repeated. Keep RUN_NAME for resume; choose a new RUN_NAME for a different experiment. Older model-only checkpoints cannot resume.


In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess, sys, os, json, shutil, hashlib, tarfile, gc

drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/Fatformer')
RUN_NAME = 'anchored_v2_run1'
CODE_REF = None  # None: previous run's pinned commit, otherwise latest hoang. Or set an exact fixed SHA.
REPO = Path('/content/fatformer-xla')
RUNTIME = Path('/content/phase7_runtime')
STAGE = Path('/content/dataset_local/phase7')
RUN_DIR = RUNTIME / 'runs' / RUN_NAME
BACKUP = DRIVE / 'checkpoint' / 'phase7' / RUN_NAME
RUNTIME.mkdir(parents=True, exist_ok=True)

def run(args, cwd=None):
    subprocess.run([str(x) for x in args], cwd=cwd or (REPO if REPO.exists() else '/content'), check=True)

def digest(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

run(['nvidia-smi'])
import torch
assert torch.cuda.is_available(), 'A CUDA runtime is required. CPU is not a training preflight PASS.'
free, total = torch.cuda.mem_get_info()
assert free >= 14 * 1024**3, f'Need >=14 GiB FREE VRAM; got {free / 1024**3:.2f}'
print(torch.cuda.get_device_name(), 'free GiB:', free / 1024**3)

## 2. Checkout reproducible code and install Python dependencies

In [ ]:
saved_ref = BACKUP / 'source_commit.txt'
ref = CODE_REF or (saved_ref.read_text().strip() if saved_ref.exists() else 'hoang')
if not REPO.exists():
    run(['git', 'clone', '-b', 'hoang', 'https://github.com/LeeVHoangtk3/fatformer-xla.git', REPO], cwd='/content')
dirty = subprocess.check_output(['git', 'status', '--porcelain', '--untracked-files=no'], cwd=REPO, text=True)
assert not dirty.strip(), 'Tracked code has local edits; preserve them before changing the checkout.'
run(['git', 'fetch', 'origin', ref])
run(['git', 'checkout', '--detach', 'FETCH_HEAD'])
SOURCE_COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
run([sys.executable, '-m', 'pip', 'install', '-q', 'ftfy', 'regex', 'tqdm', 'pyyaml', 'timm', 'einops', 'pytest', 'numpy', 'Pillow', 'scikit-learn', 'pandas', 'matplotlib'])
help_text = subprocess.check_output([sys.executable, '-m', 'src.training.train_crossdomain', '--help'], cwd=REPO, text=True)
assert all(flag in help_text for flag in ['--resume_from', '--benchmark_steps', '--data_root']), 'Stale code: checkout a commit containing the Task 7.11 fixes first.'
import yaml
print('Pinned source:', SOURCE_COMMIT)

## 3. Verify archive checksum, stage images and create a runtime config
The original audit/manifest/data config stay byte-for-byte intact. Paths are resolved with data_root; the manifest is not rewritten.

In [ ]:
archive = DRIVE / 'phase7' / 'train_crossdomain_v2.tar'
checksum = Path(str(archive) + '.sha256')
assert archive.is_file() and checksum.is_file(), 'Need both the audited archive and its original .sha256 sidecar.'
expected = checksum.read_text().split()[0]
assert len(expected) == 64 and all(c in '0123456789abcdef' for c in expected.lower())
STAGE.mkdir(parents=True, exist_ok=True)
local_archive = STAGE / archive.name
if not local_archive.exists() or digest(local_archive) != expected:
    shutil.copy2(archive, local_archive)
assert digest(local_archive) == expected, 'Archive SHA256 mismatch; do not extract or train.'
marker = STAGE / '.extracted_sha256'
if not marker.exists() or marker.read_text().strip() != expected:
    assert sys.version_info >= (3, 12), 'Use a runtime with tarfile data extraction filters.'
    with tarfile.open(local_archive) as bundle:
        bundle.extractall(STAGE, filter='data')
    marker.write_text(expected)

assets = RUNTIME / 'pretrained'
assets.mkdir(exist_ok=True)
for name in ('ViT-L-14.pt', 'fatformer_4class_ckpt.pth'):
    source = DRIVE / 'pretrained' / name
    assert source.is_file(), f'Missing model asset: {source}'
    target = assets / name
    if not target.exists() or digest(target) != digest(source):
        shutil.copy2(source, target)
    assert digest(target) == digest(source)

previous_config = BACKUP / 'run_config.yaml'
base = previous_config if previous_config.exists() else REPO / 'configs/train_phase7_anchored_v2.yaml'
config = yaml.safe_load(base.read_text())
config['dataset'].update(manifest=str(STAGE / 'DATASET/manifests/data_manifest_v2.json'),
                         isolation_report=str(STAGE / 'DATASET/logs/phase7/data_isolation_report.json'),
                         audit_config=str(STAGE / 'configs/phase7_data_v2.yaml'),
                         data_root=str(STAGE), train_dir='', val_dir='', val_split='val_select',
                         degradation_config=str(REPO / 'configs/degradation_v2.yaml'))
config['model'].update(clip_path=str(assets / 'ViT-L-14.pt'),
                       checkpoint_init=str(assets / 'fatformer_4class_ckpt.pth'),
                       teacher_checkpoint=str(assets / 'fatformer_4class_ckpt.pth'))
config['training'].update(device='cuda', save_dir=str(RUN_DIR), backup_dir=str(BACKUP),
                          history_csv=str(RUN_DIR / 'train_history.csv'),
                          calibration_file=str(REPO / 'DATASET/logs/phase7/calibration/baseline.json'))
config['training'].pop('resume_from', None)
CONFIG = RUNTIME / 'train_runtime.yaml'
CONFIG.write_text(yaml.safe_dump(config, sort_keys=False))
print('mini-batch:', config['training']['batch_size'], 'grad accumulation:', config['training']['grad_accum_steps'])
print('Effective batch:', config['training']['batch_size'] * config['training']['grad_accum_steps'])

## 4. Required preflight on this GPU runtime
Every image is decoded and SHA256-checked. Missing images, stale evidence, failed tests, canonical-load errors or insufficient free VRAM stop execution.

In [ ]:
PREFLIGHT = RUNTIME / 'preflight_train.json'
run([sys.executable, 'tools/preflight_phase7_train.py', '--config', CONFIG, '--output', PREFLIGHT])
preflight = json.loads(PREFLIGHT.read_text())
assert preflight['ready_to_train'] is True

## 5. Dedicated 100-mini-batch benchmark
Separate process and output. No training checkpoints/CSV are written. Estimates cover training steps only; validation and Drive backup add time. A resumed run retains its saved epoch/scheduler horizon.

In [ ]:
RESUMING = (RUN_DIR / 'checkpoint_latest.pth').exists() or (BACKUP / 'checkpoint_latest.pth').exists()
BENCHMARK = RUNTIME / 'benchmark' / 'train_benchmark_100steps.json'
if not RESUMING:
    run([sys.executable, '-m', 'src.training.train_crossdomain', '--config', CONFIG,
         '--benchmark_steps', '100', '--benchmark_warmup', '5', '--output', BENCHMARK,
         '--save_dir', RUNTIME / 'benchmark' / 'scratch'])
    measurement = json.loads(BENCHMARK.read_text())
    assert measurement['evidence_level'] == 'measured' and measurement['device'].startswith('cuda')
    print(json.dumps(measurement, indent=2))
    if 'T4' in torch.cuda.get_device_name() and measurement['est_total_minutes'] > 300:
        config['training']['epochs'] = 4
        if measurement['est_minutes_per_epoch'] * 4 > 300:
            raise RuntimeError('Even 4 epochs exceed the 300-minute training estimate. Review GPU/budget before training.')
        CONFIG.write_text(yaml.safe_dump(config, sort_keys=False))
else:
    print('Resume detected: benchmark skipped and the original epoch horizon retained.')

## 6. Train or resume from the last completed epoch
Latest is the full resume state. Official best and best-effort are model-only inference artifacts. Legacy model-only latest files are rejected. An interrupted epoch is replayed; no claim of cross-hardware bit-exact GPU execution is made.

In [ ]:
# Recheck code identity and final runtime configuration before training.
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip() == SOURCE_COMMIT
run(['git', 'diff', '--exit-code', 'HEAD'])
preflight = json.loads(PREFLIGHT.read_text())
if preflight['config_sha256'] != digest(CONFIG) or not preflight['ready_to_train']:
    run([sys.executable, 'tools/preflight_phase7_train.py', '--config', CONFIG, '--output', PREFLIGHT])
assert json.loads(PREFLIGHT.read_text())['ready_to_train'] is True
RUN_DIR.mkdir(parents=True, exist_ok=True)
BACKUP.mkdir(parents=True, exist_ok=True)
latest = RUN_DIR / 'checkpoint_latest.pth'
remote = BACKUP / 'checkpoint_latest.pth'
if not latest.exists() and not remote.exists():
    assert BENCHMARK.exists(), 'Run the measured benchmark successfully before starting a new run.'
    measured = json.loads(BENCHMARK.read_text())
    assert measured['evidence_level'] == 'measured' and measured['gpu_name'] == torch.cuda.get_device_name()
    assert measured['context']['source_sha256'] == json.loads(PREFLIGHT.read_text())['source_sha256']
    for key in ('manifest', 'degradation_config'):
        context_key = 'train_manifest_sha256' if key == 'manifest' else key + '_sha256'
        assert measured['context'][context_key] == digest(config['dataset'][key]), 'Inputs changed; rerun benchmark.'
    assert measured['context']['calibration_file_sha256'] == digest(config['training']['calibration_file'])
    for section, keys in [('training', ('batch_size', 'grad_accum_steps', 'use_amp', 'learning_rate', 'anchor_learning_rate')),
                          ('loss', ('weight_ce', 'weight_ce_deg', 'alpha_kd', 'beta_cons', 'weight_feat', 'weight_guard', 'temperature'))]:
        for key in keys:
            assert measured['context'][key] == config[section][key], 'Hyperparameters changed; rerun benchmark.'
    if 'T4' in torch.cuda.get_device_name():
        assert measured['est_minutes_per_epoch'] * config['training']['epochs'] <= 300, 'Review the GPU budget first.'
if not latest.exists() and remote.exists():
    remote_checksum = Path(str(remote) + '.sha256')
    assert remote_checksum.exists(), 'Legacy/unverified latest: use a new RUN_NAME or a full verified resume state.'
    temporary = Path(str(latest) + '.download')
    shutil.copy2(remote, temporary)
    assert digest(temporary) == remote_checksum.read_text().strip(), 'Resume checksum mismatch.'
    os.replace(temporary, latest)
    for name in (config['training']['save_name'], 'best_effort.pth'):
        if (BACKUP / name).exists():
            shutil.copy2(BACKUP / name, RUN_DIR / name)

(BACKUP / 'run_config.yaml').write_text(CONFIG.read_text())
(BACKUP / 'source_commit.txt').write_text(SOURCE_COMMIT + '\n')
if BENCHMARK.exists():
    shutil.copy2(BENCHMARK, BACKUP / BENCHMARK.name)
shutil.copy2(PREFLIGHT, BACKUP / PREFLIGHT.name)
command = [sys.executable, '-m', 'src.training.train_crossdomain', '--config', CONFIG]
if latest.exists():
    command += ['--resume_from', latest]
run(command)
print('Run process completed. Review the groups/guard gates; this does not automatically pass G7.2.')

## 7. Review this run and export its convergence chart

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
csv_path = RUN_DIR / 'val_groups.csv'
assert csv_path.exists(), 'No validated epoch log was produced.'
df = pd.read_csv(csv_path)
print(df.to_string(index=False))
fig, axes = plt.subplots(2, 1, figsize=(10, 10))
ax = axes[0]
for key in ('clean_r', 'clean_f', 'q30_r', 'q30_f', 'worst'):
    ax.plot(df['epoch'], df[key] * 100, marker='o', label=key)
ax.axhline(90, color='gray', linestyle='--', label='Clean Real requirement')
ax.set(xlabel='Completed epoch', ylabel='Recall / worst group (%)', title='Phase 7.11')
ax.legend(); ax.grid(alpha=.2)
history = pd.read_csv(RUN_DIR / 'train_history.csv')
for key in ('loss_total', 'loss_ce', 'loss_kd', 'loss_cons', 'weighted_feat', 'weighted_guard'):
    if key in history:
        axes[1].plot(history['epoch'], history[key], marker='o', label=key)
axes[1].set(xlabel='Completed epoch', ylabel='Loss', title='Training loss components')
axes[1].legend(); axes[1].grid(alpha=.2)
fig.tight_layout()
chart = RUN_DIR / 'm7_convergence.png'
fig.savefig(chart, dpi=160, bbox_inches='tight')
plt.show()
shutil.copy2(chart, BACKUP / chart.name)
if not (RUN_DIR / config['training']['save_name']).exists():
    print('No official best is present. best_effort is for analysis only.')
print('Artifacts:', BACKUP)